In [1]:
import pandas as pd
import plotly.express as px
import streamlit as st

In [2]:
df_transactions = pd.read_csv('df/transactions.csv')
df_cards = pd.read_csv('df/cards.csv')
df_users = pd.read_csv('df/users.csv')

df_users.head()

,user_id,registration_date,age,activity_segment,financial_awareness,has_salom,has_osmon,has_virtual_uzcard,has_virtual_humo,other_bank_cards_count
0,1,2025-08-30,20,active,medium,False,False,True,False,0
1,2,2024-02-28,62,active,medium,True,False,True,True,0
2,3,2025-08-03,37,active,low,False,False,True,False,0
3,4,2024-10-08,27,medium,high,False,True,True,False,1
4,5,2024-11-05,24,super_active,high,True,False,False,True,1


In [3]:
df_users.info()
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   user_id                 10000 non-null  int64 
 1   registration_date       10000 non-null  object
 2   age                     10000 non-null  int64 
 3   activity_segment        10000 non-null  object
 4   financial_awareness     10000 non-null  object
 5   has_salom               10000 non-null  bool  
 6   has_osmon               10000 non-null  bool  
 7   has_virtual_uzcard      10000 non-null  bool  
 8   has_virtual_humo        10000 non-null  bool  
 9   other_bank_cards_count  10000 non-null  int64 
dtypes: bool(4), int64(3), object(3)
memory usage: 507.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220000 entries, 0 to 219999
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 

In [4]:
df_users.isna().sum()

user_id                   0
registration_date         0
age                       0
activity_segment          0
financial_awareness       0
has_salom                 0
has_osmon                 0
has_virtual_uzcard        0
has_virtual_humo          0
other_bank_cards_count    0
dtype: int64

In [5]:
df_transactions.describe()

,transaction_id,user_id,amount,fee_amount
count,220000.000000,220000.000000,2.200000e+05,220000.000000
mean,110000.500000,4997.246073,3.003947e+05,2242.329468
std,63508.673948,2884.560638,3.002518e+05,4645.917911
min,1.000000,1.000000,2.490000e+00,0.000000
25%,55000.750000,2501.000000,8.626711e+04,0.000000
50%,110000.500000,4994.000000,2.077811e+05,0.000000
75%,165000.250000,7492.250000,4.167733e+05,2423.032500
max,220000.000000,10000.000000,3.492974e+06,63644.090000


In [6]:
df_transactions['date'] = pd.to_datetime(df_transactions['transaction_date'])

In [7]:
dau = df_transactions.groupby(df_transactions['date'].dt.date)['user_id'].nunique()
dau

date
2024-01-01    279
2024-01-02    297
2024-01-03    276
2024-01-04    302
2024-01-05    292
             ... 
2025-12-27    279
2025-12-28    322
2025-12-29    307
2025-12-30    265
2025-12-31    306
Name: user_id, Length: 731, dtype: int64

In [8]:
mau = df_transactions.groupby(df_transactions['date'].dt.to_period('M'))['user_id'].nunique()
mau

date
2024-01    6114
2024-02    5740
2024-03    6148
2024-04    5969
2024-05    6014
2024-06    5931
2024-07    6038
2024-08    5977
2024-09    5961
2024-10    6089
2024-11    5987
2024-12    6071
2025-01    5987
2025-02    5654
2025-03    6075
2025-04    5940
2025-05    6048
2025-06    5965
2025-07    6091
2025-08    6047
2025-09    5941
2025-10    6091
2025-11    5937
2025-12    6161
Freq: M, Name: user_id, dtype: int64

In [9]:
df_transactions['amount'].mean()

np.float64(300394.73732099996)

In [10]:
df_transactions.groupby('transaction_type')['amount'].mean()

transaction_type
cash_withdrawal    301824.120369
credit_payment     301193.997737
p2p_transfer       299917.541160
payment            300447.015122
Name: amount, dtype: float64

In [11]:
df_transactions['fee_amount'].sum()

np.float64(493312482.9)

In [12]:
df_transactions.groupby('route_type')['fee_amount'].sum()

route_type
direct       493312482.9
via_salom            0.0
Name: fee_amount, dtype: float64

In [13]:
fig = px.bar(
    df_transactions.groupby('transaction_type').size().reset_index(name='count'),
    x='transaction_type',
    y='count',
    title='Transaction by type'
)

fig.show()

In [14]:
fig = px.line (
    dau,
    title='Daily Active Users'
)

fig.show()

In [15]:
salom = df_transactions[df_transactions['route_type'] == 'via_salom']

salom_count = len(salom)
all_tx = len(df_transactions)

salom_share = salom_count / all_tx * 100

salom_share

12.46409090909091

In [16]:
fees = (
    df_transactions
    .groupby('route_type')['fee_amount']
    .sum()
    .reset_index()
)

In [17]:
fig = px.bar(
    fees,
    x='route_type',
    y='fee_amount',
    title='Total Fees by Transfer Route'
)

fig.show()

In [18]:
bank_usage = (
    df_cards
    .groupby('bank_name')['user_id']
    .nunique()
    .reset_index(name='users')
)

In [19]:
fig = px.pie(
    bank_usage,
    names='bank_name',
    values='users',
    title='Connected Cards by Bank'
)

fig.show()

In [20]:
top_users = (
    df_transactions
    .groupby('user_id')['amount']
    .sum()
    .reset_index()
    .sort_values('amount', ascending=False)
    .head(10)
)

In [21]:
fig = px.bar(
    top_users,
    x='user_id',
    y='amount',
    title='Top 10 Users by Transaction Volume'
)

fig.show()

In [22]:
salom_usage = (
    df_transactions
    .groupby('route_type')
    .size()
    .reset_index(name='count')
)

In [23]:
fig = px.pie(
    salom_usage,
    names='route_type',
    values='count',
    title='Transfer Routes Distribution'
)

fig.show()